In [1]:
from pathlib import Path
import zipfile
import shutil
import hashlib
import re
import os
import pandas as pd

In [4]:
#get the file 


CURRENT_DIR = Path.cwd()

print("Jupyter folder:")
print(CURRENT_DIR)

print("\nZIP files found:")

for p in CURRENT_DIR.glob("*.zip"):
    print(p.name)
zip_files = list(CURRENT_DIR.glob("*.zip"))

if not zip_files:
    raise FileNotFoundError("No ZIP file found in the Jupyter folder.")

if len(zip_files) > 1:
    print("Multiple ZIP files found:")
    for i, p in enumerate(zip_files):
        print(i, p.name)
else:
    print("Using:", zip_files[0].name)
ZIP_PATH = zip_files[0]

print("ZIP:", ZIP_PATH)
print(
    "ZIP size:",
    round(ZIP_PATH.stat().st_size / (1024**2), 2),
    "MB"
)

Jupyter folder:
C:\Users\ASUS\Documents

ZIP files found:
dataset.zip
Using: dataset.zip
ZIP: C:\Users\ASUS\Documents\dataset.zip
ZIP size: 772.19 MB


In [5]:
#Extract 
EXTRACT_ROOT = CURRENT_DIR / "original_dataset"

EXTRACT_ROOT.mkdir(exist_ok=True)

print("Extracting...")

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_ROOT)

print("Extraction complete.")
print("Extracted to:", EXTRACT_ROOT)

all_files = [
    p for p in EXTRACT_ROOT.rglob("*")
    if p.is_file()
]

print("Total files:", len(all_files))

total_size_gb = sum(
    p.stat().st_size for p in all_files
) / (1024 ** 3)

print(f"Total extracted size: {total_size_gb:.2f} GB")

Extracting...
Extraction complete.
Extracted to: C:\Users\ASUS\Documents\original_dataset
Total files: 15574
Total extracted size: 0.80 GB


In [6]:
#start cleaning - 1. remove unsupported 
SUPPORTED_AUDIO_EXTENSIONS = {
    ".wav",
    ".mp3",
    ".flac",
    ".ogg"
}

audio_files = [
    p for p in EXTRACT_ROOT.rglob("*")
    if p.is_file()
    and p.suffix.lower() in SUPPORTED_AUDIO_EXTENSIONS
]

print("Audio files found:", len(audio_files))

Audio files found: 15552


In [29]:
PROJECT_ROOT = Path.cwd()

VALIDATION_DIR = PROJECT_ROOT / "validation_reports"

MANIFEST_PATH = VALIDATION_DIR / "dataset_manifest.csv"
FILE_DUPLICATES_PATH = VALIDATION_DIR / "file_duplicates.csv"
METADATA_DUPLICATES_PATH = VALIDATION_DIR / "metadata_duplicates.csv"

print("Project root:", PROJECT_ROOT)
print("Validation directory:", VALIDATION_DIR)

Project root: C:\Users\ASUS\Documents
Validation directory: C:\Users\ASUS\Documents\validation_reports


In [30]:
for path in [
    MANIFEST_PATH,
    FILE_DUPLICATES_PATH,
    METADATA_DUPLICATES_PATH
]:
    print(
        "✓" if path.exists() else "✗",
        path
    )

✓ C:\Users\ASUS\Documents\validation_reports\dataset_manifest.csv
✓ C:\Users\ASUS\Documents\validation_reports\file_duplicates.csv
✓ C:\Users\ASUS\Documents\validation_reports\metadata_duplicates.csv


In [31]:
manifest = pd.read_csv(MANIFEST_PATH)

print("Validation manifest loaded.")
print("Records:", len(manifest))

print("\nColumns:")
print(manifest.columns.tolist())

Validation manifest loaded.
Records: 15574

Columns:
['file_name', 'absolute_path', 'relative_path', 'extension', 'size_bytes', 'source', 'species', 'species_label', 'format_status', 'readable', 'duration', 'sample_rate', 'channels', 'issue']


In [32]:
file_duplicates = pd.read_csv(
    FILE_DUPLICATES_PATH
)

print(
    "File duplicate records:",
    len(file_duplicates)
)

print(
    file_duplicates.columns.tolist()
)

metadata_duplicates = pd.read_csv(
    METADATA_DUPLICATES_PATH
)

print(
    "Metadata duplicate records:",
    len(metadata_duplicates)
)

File duplicate records: 10
['file_name', 'absolute_path', 'relative_path', 'extension', 'size_bytes', 'source', 'species', 'species_label', 'sha256']
Metadata duplicate records: 0


In [33]:
clean_manifest = manifest.copy()

clean_manifest["cleaning_action"] = "keep"
clean_manifest["cleaning_reason"] = ""

In [34]:
#remove empty
empty_mask = (
    clean_manifest["size_bytes"].fillna(0) == 0
)

clean_manifest.loc[
    empty_mask,
    "cleaning_action"
] = "remove"

clean_manifest.loc[
    empty_mask,
    "cleaning_reason"
] = "empty_file"

In [35]:
#remove unreadable
unreadable_mask = (
    clean_manifest["readable"] != True
)

clean_manifest.loc[
    unreadable_mask,
    "cleaning_action"
] = "remove"

clean_manifest.loc[
    unreadable_mask,
    "cleaning_reason"
] = "unreadable"

In [36]:
#remove files under 1 second 
too_short_mask = (
    clean_manifest["duration"].notna()
    & (clean_manifest["duration"] < 1.0)
)

clean_manifest.loc[
    too_short_mask,
    "cleaning_action"
] = "remove"

clean_manifest.loc[
    too_short_mask,
    "cleaning_reason"
] = "too_short_less_than_1_second"

In [37]:
#standadrise species names 
clean_manifest["species_label_original"] = (
    clean_manifest["species_label"]
)
def standardise_species_name(name):

    if pd.isna(name):
        return "unknown"

    name = str(name).strip()

    # Underscores → spaces
    name = name.replace("_", " ")

    # Multiple spaces → one space
    name = re.sub(r"\s+", " ", name)

    return name

clean_manifest["species_label"] = (
    clean_manifest["species_label"]
    .apply(standardise_species_name)
)

In [38]:
species_changes = (
    clean_manifest[
        clean_manifest["species_label_original"]
        != clean_manifest["species_label"]
    ][
        [
            "species_label_original",
            "species_label"
        ]
    ]
    .drop_duplicates()
    .sort_values("species_label")
)

species_changes

,species_label_original,species_label
12514,acanthiza_chrysorrhoa,acanthiza chrysorrhoa
14437,acanthiza_lineata,acanthiza lineata
983,acanthiza_nana,acanthiza nana
13759,acanthiza_pusilla,acanthiza pusilla
10191,acanthiza_reguloides,acanthiza reguloides
...,...,...
11133,trichosurus_vulpecula,trichosurus vulpecula
0,uperoleia_altissima,uperoleia altissima
134,uperoleia_mimula,uperoleia mimula
12612,vanellus_miles,vanellus miles


In [39]:
#duplicates keep the bucket 1 and remove bucket 3 
file_duplicates["is_bucket_1"] = (
    file_duplicates["relative_path"]
    .str.contains(
        "bucket_1",
        case=False,
        na=False
    )
)

file_duplicates["is_bucket_3"] = (
    file_duplicates["relative_path"]
    .str.contains(
        "bucket_3",
        case=False,
        na=False
    )
)

bucket1_hashes = set(
    file_duplicates.loc[
        file_duplicates["is_bucket_1"],
        "sha256"
    ].dropna()
)

bucket3_to_remove = file_duplicates[
    file_duplicates["is_bucket_3"]
    & file_duplicates["sha256"].isin(
        bucket1_hashes
    )
].copy()

print(
    "Bucket 3 exact duplicates to remove:",
    len(bucket3_to_remove)
)

Bucket 3 exact duplicates to remove: 5


In [40]:
bucket3_paths = set(
    bucket3_to_remove["relative_path"]
)
bucket3_mask = (
    clean_manifest["relative_path"]
    .isin(bucket3_paths)
)
clean_manifest.loc[
    bucket3_mask,
    "cleaning_action"
] = "remove"

clean_manifest.loc[
    bucket3_mask,
    "cleaning_reason"
] = "exact_duplicate_keep_bucket1"

In [41]:
# Remove unsupported formats from the cleaning manifest

SUPPORTED_AUDIO_EXTENSIONS = {
    ".wav",
    ".mp3",
    ".flac",
    ".ogg"
}

unsupported_mask = ~(
    clean_manifest["extension"]
    .str.lower()
    .isin(SUPPORTED_AUDIO_EXTENSIONS)
)

clean_manifest.loc[
    unsupported_mask,
    "cleaning_action"
] = "remove"

clean_manifest.loc[
    unsupported_mask,
    "cleaning_reason"
] = "unsupported_extension"

In [42]:
#chekc removal summary 
removal_summary = (
    clean_manifest[
        clean_manifest["cleaning_action"] == "remove"
    ]
    .groupby("cleaning_reason")
    .size()
    .reset_index(name="file_count")
    .sort_values(
        "file_count",
        ascending=False
    )
)

removal_summary

print(
    "Original files:",
    len(clean_manifest)
)

print(
    "Files to remove:",
    (
        clean_manifest["cleaning_action"]
        == "remove"
    ).sum()
)

print(
    "Files to keep:",
    (
        clean_manifest["cleaning_action"]
        == "keep"
    ).sum()
)

Original files: 15574
Files to remove: 747
Files to keep: 14827


In [43]:
print("Manifest total:", len(clean_manifest))

print("\nExpected individual categories:")
print(
    "Under 1 sec:",
    (
        clean_manifest["duration"].notna()
        & (clean_manifest["duration"] < 1)
    ).sum()
)

print(
    "Unsupported:",
    (
        ~clean_manifest["extension"]
        .str.lower()
        .isin(SUPPORTED_AUDIO_EXTENSIONS)
    ).sum()
)

print(
    "Unreadable:",
    (clean_manifest["readable"] != True).sum()
)

print(
    "Empty:",
    (clean_manifest["size_bytes"].fillna(0) == 0).sum()
)

print(
    "Currently marked remove:",
    (
        clean_manifest["cleaning_action"]
        == "remove"
    ).sum()
)

Manifest total: 15574

Expected individual categories:
Under 1 sec: 722
Unsupported: 22
Unreadable: 22
Empty: 0
Currently marked remove: 747


In [44]:
short_mask = (
    clean_manifest["duration"].notna()
    & (clean_manifest["duration"] < 1)
)

unsupported_mask = ~(
    clean_manifest["extension"]
    .str.lower()
    .isin(SUPPORTED_AUDIO_EXTENSIONS)
)

duplicate_mask = (
    clean_manifest["relative_path"]
    .isin(bucket3_paths)
)

print("Short:", short_mask.sum())
print("Unsupported:", unsupported_mask.sum())
print("Bucket 3 duplicates:", duplicate_mask.sum())

print("\nShort + unsupported:",
      (short_mask & unsupported_mask).sum())

print("Short + duplicate:",
      (short_mask & duplicate_mask).sum())

print("Unsupported + duplicate:",
      (unsupported_mask & duplicate_mask).sum())

print("All three:",
      (short_mask & unsupported_mask & duplicate_mask).sum())

Short: 722
Unsupported: 22
Bucket 3 duplicates: 5

Short + unsupported: 0
Short + duplicate: 2
Unsupported + duplicate: 0
All three: 0


In [45]:
short_and_duplicate = clean_manifest[
    short_mask & duplicate_mask
].copy()

short_and_duplicate[
    [
        "file_name",
        "relative_path",
        "species_label",
        "duration",
        "extension",
        "size_bytes",
        "cleaning_action",
        "cleaning_reason"
    ]
]

,file_name,relative_path,species_label,duration,extension,size_bytes,cleaning_action,cleaning_reason
14389,project_echo_bucket_3__Ground_Parrot (14).wav,GCP/Pezoporus wallicus/project_echo_bucket_3__...,pezoporus wallicus,0.35,.wav,61784,remove,exact_duplicate_keep_bucket1
14425,project_echo_bucket_3__Ground_Parrot (11).wav,GCP/Pezoporus wallicus/project_echo_bucket_3__...,pezoporus wallicus,0.95,.wav,167624,remove,exact_duplicate_keep_bucket1


In [46]:
##verfications 
short_kept = clean_manifest[
    (
        clean_manifest["duration"].notna()
        & (clean_manifest["duration"] < 1.0)
        & (clean_manifest["cleaning_action"] == "keep")
    )
]

print(
    "Short files still being kept:",
    len(short_kept)
)

unsupported_kept = clean_manifest[
    (
        ~clean_manifest["extension"]
        .str.lower()
        .isin(SUPPORTED_AUDIO_EXTENSIONS)
    )
    &
    (
        clean_manifest["cleaning_action"] == "keep"
    )
]

print(
    "Unsupported files still being kept:",
    len(unsupported_kept)
)

duplicate_kept = clean_manifest[
    duplicate_mask
    &
    (
        clean_manifest["cleaning_action"] == "keep"
    )
]

print(
    "Bucket-3 duplicates still being kept:",
    len(duplicate_kept)
)

Short files still being kept: 0
Unsupported files still being kept: 0
Bucket-3 duplicates still being kept: 0


In [47]:
#final clenaed manifest 
final_manifest = clean_manifest[
    clean_manifest["cleaning_action"] == "keep"
].copy()

print(
    "Final cleaned records:",
    len(final_manifest)
)

Final cleaned records: 14827


In [49]:
species_distribution = (
    final_manifest
    .groupby("species_label")
    .size()
    .reset_index(name="file_count")
)

species_distribution

,species_label,file_count
0,acanthiza chrysorrhoa,62
1,acanthiza lineata,58
2,acanthiza nana,258
3,acanthiza pusilla,480
4,acanthiza reguloides,268
...,...,...
117,trichosurus vulpecula,14
118,uperoleia altissima,130
119,uperoleia mimula,90
120,vanellus miles,108


In [53]:
# Look for the GCP folder in the current project
gcp_folders = list(
    PROJECT_ROOT.rglob("GCP")
)

for folder in gcp_folders:
    print(folder)

C:\Users\ASUS\Documents\original_dataset\dataset\GCP


In [54]:
ORIGINAL_DATASET_ROOT = (
    gcp_folders[0].parent
)

print(ORIGINAL_DATASET_ROOT)

missing_files = []

for _, row in final_manifest.iterrows():

    source = (
        ORIGINAL_DATASET_ROOT
        / row["relative_path"]
    )

    if not source.exists():
        missing_files.append(str(source))

print("Missing files:", len(missing_files))

print(
    "Original dataset exists:",
    ORIGINAL_DATASET_ROOT.exists()
)

C:\Users\ASUS\Documents\original_dataset\dataset
Missing files: 0
Original dataset exists: True


In [51]:
#create clened data set 
CLEANED_ROOT = (
    PROJECT_ROOT / "cleaned_dataset"
)

CLEANED_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Cleaned dataset:",
    CLEANED_ROOT
)

Cleaned dataset: C:\Users\ASUS\Documents\cleaned_dataset


In [55]:
for _, row in final_manifest.iterrows():

    source = (
        ORIGINAL_DATASET_ROOT
        / row["relative_path"]
    )

    species_folder = (
        CLEANED_ROOT
        / row["species_label"]
    )

    destination = (
        species_folder
        / row["file_name"]
    )

    species_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    shutil.copy2(
        source,
        destination
    )

In [58]:
#verify and rpeorts 
# Verify and report missing cleaned files

missing_cleaned = []

for _, row in final_manifest.iterrows():

    cleaned_path = (
        CLEANED_ROOT
        / row["species_label"]
        / row["file_name"]
    )

    if not cleaned_path.exists():
        missing_cleaned.append(str(cleaned_path))

print("Missing cleaned files:", len(missing_cleaned))

print(
    "Cleaned dataset exists:",
    CLEANED_ROOT.exists()
)

Missing cleaned files: 0
Cleaned dataset exists: True


In [59]:
CLEANING_REPORTS = (
    PROJECT_ROOT / "cleaning_reports"
)

CLEANING_REPORTS.mkdir(
    parents=True,
    exist_ok=True
)

clean_manifest.to_csv(
    CLEANING_REPORTS
    / "cleaning_manifest.csv",
    index=False
)

final_manifest.to_csv(
    CLEANING_REPORTS
    / "final_cleaned_manifest.csv",
    index=False
)

removed_files = clean_manifest[
    clean_manifest["cleaning_action"] == "remove"
].copy()

removed_files.to_csv(
    CLEANING_REPORTS
    / "removed_files.csv",
    index=False
)

removal_summary.to_csv(
    CLEANING_REPORTS
    / "removal_summary.csv",
    index=False
)

species_distribution.to_csv(
    CLEANING_REPORTS
    / "species_distribution_after_cleaning.csv"
)

In [ ]:
#Sprint 2
#Raveesha Somawansa
#s225559348